In [ ]:
DATALAKE_BUCKET = "gksdatalake"

In [ ]:
import os

import boto3
from botocore.exceptions import BotoCoreError, ClientError, NoCredentialsError

# boto3 automatically reads:
# AWS_ACCESS_KEY_ID
# AWS_SECRET_ACCESS_KEY
# Optional: AWS_SESSION_TOKEN

required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
missing = [name for name in required if not os.getenv(name)]

if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}")

In [ ]:
session = boto3.Session(region_name="us-east-1")
s3 = session.client("s3")

try:
    response = s3.list_buckets()

    print("Available S3 buckets:")
    for bucket in response.get("Buckets", []):
        print(f"- {bucket['Name']}")

except NoCredentialsError:
    print("AWS credentials were not found.")
except ClientError as exc:
    print(f"AWS API error: {exc}")
except BotoCoreError as exc:
    print(f"AWS SDK error: {exc}")

In [ ]:
# now upload the public key to AWS

from pathlib import Path
from botocore.exceptions import ClientError

KEY_NAME = "ec2emrkey"
PUBLIC_KEY_PATH = Path.home() / ".ssh" / "ec2emrkey.pem.pub"

ec2 = session.client("ec2")

if not PUBLIC_KEY_PATH.exists():
    raise FileNotFoundError(
        f"Public key not found: {PUBLIC_KEY_PATH}\n"
        "Generate it first using ssh-keygen."
    )

try:
    response = ec2.import_key_pair(
        KeyName=KEY_NAME,
        PublicKeyMaterial=PUBLIC_KEY_PATH.read_bytes()
    )

    print("Key imported successfully")
    print("Key name:", response["KeyName"])
    print("Key fingerprint:", response["KeyFingerprint"])

except ClientError as exc:
    error_code = exc.response["Error"]["Code"]

    if error_code == "InvalidKeyPair.Duplicate":
        print(
            f"A key named {KEY_NAME!r} already exists in "
            f"{session.region_name}; no import was performed."
        )
    else:
        raise

In [ ]:
import botocore.exceptions

s3 = session.client("s3")

try:
    # Checks whether the bucket exists and is accessible
    s3.head_bucket(Bucket=DATALAKE_BUCKET)
    print(f"Bucket already exists: s3://{DATALAKE_BUCKET}")

except botocore.exceptions.ClientError as error:
    error_code = error.response["Error"]["Code"]

    if error_code == "404":
        # us-east-1 must not use CreateBucketConfiguration
        s3.create_bucket(Bucket=DATALAKE_BUCKET)
        print(f"Bucket created: s3://{DATALAKE_BUCKET}")
    else:
        # Includes 403 when the globally unique name belongs to someone else
        raise

In [ ]:
import boto3
import urllib.request
from botocore.exceptions import ClientError

session = boto3.Session(region_name="us-east-1")
ec2 = session.client("ec2")

# Get the Pluralsight VM's current public outbound IP
current_ip = urllib.request.urlopen(
    "https://checkip.amazonaws.com",
    timeout=10
).read().decode().strip()

cidr_ip = f"{current_ip}/32"

# Find the default VPC in us-east-1
vpcs = ec2.describe_vpcs(
    Filters=[
        {"Name": "is-default", "Values": ["true"]}
    ]
)["Vpcs"]

if not vpcs:
    raise RuntimeError("No default VPC found in us-east-1")

default_vpc_id = vpcs[0]["VpcId"]

# Find the default security group belonging to that VPC
security_groups = ec2.describe_security_groups(
    Filters=[
        {"Name": "vpc-id", "Values": [default_vpc_id]},
        {"Name": "group-name", "Values": ["default"]}
    ]
)["SecurityGroups"]

if not security_groups:
    raise RuntimeError("Default security group not found")

security_group_id = security_groups[0]["GroupId"]

print("Current public IP:", current_ip)
print("Default VPC:", default_vpc_id)
print("Default security group:", security_group_id)

# Allow SSH only from this VM's current public IP
try:
    ec2.authorize_security_group_ingress(
        GroupId=security_group_id,
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": 22,
                "ToPort": 22,
                "IpRanges": [
                    {
                        "CidrIp": cidr_ip,
                        "Description": "Pluralsight VM"
                    }
                ]
            }
        ]
    )

    print(f"Added SSH access from {cidr_ip}")

except ClientError as error:
    if error.response["Error"]["Code"] == "InvalidPermission.Duplicate":
        print(f"Rule already exists for {cidr_ip}")
    else:
        raise